In [45]:
import pandas as pd

df_gmv = pd.read_csv("./test/gmv_clean_sales_infor.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./test/leads_clean_with_note_phone.csv", encoding='utf-8-sig')
df_unmatched = pd.read_csv("./test/gmv_unmatched.csv", encoding='utf-8-sig', 
                           dtype={"Phone_clean": str, "Phone_extracted_from_note": str})

In [46]:
df_unmatched.columns

Index(['bank day', 'bank time', 'Gateway', 'User Name', 'Phone', 'UID',
       'Package', 'Fixed/ non-fixed', 'Pay Time', 'Real Pay(VND)', 'GMV (RMB)',
       'Payment Method', 'Type', 'Sales', 'Note', 'Month of payment',
       'Unnamed: 16', 'TEAM', 'Full Price(VND)', '采购价格 \n（套餐配置）', '渠道号',
       '首次申请试听日期', '首次申请试听渠道号', '首次购课时间', 'Source', '已批工单', '总 B (被推荐） 课数',
       'A (推荐人） 课数 (送PH)', 'A', 'A (推荐人）课数 (送UK)', 'B', '实际卖课单价 VND',
       '实际卖课单价 RMB', 'PH/UK', '采购价格（包括转介绍赠送课)', 'Nguồn', 'UID_clean',
       'Phone_clean', 'Sales_Abbr', 'Sales_infor', 'Đầu Ngày', 'Họ và tên',
       'Người trực', 'Số điện thoại', 'Tuổi', 'Quốc Gia', 'Note_lead', 'TVTS',
       'Ad_id', 'Nguồn_lead', 'Quốc gia \nQuảng cáo', 'Mẫu Quảng cáo',
       'Mã CRM', 'Sale sau phân', 'Link FB', 'UID_lead', 'TT', 'Trạng thái',
       'Ngày lên học thử', 'Current Binded Sale',
       'Sales in latest system-weighted allocation',
       'Latest system-weighted Allocation Time', 'Allocate reason',
       'Tên sal

In [51]:
df_unmatched["Phone_clean"]

0       84908224672
1       84909517679
2       84389970625
3       84938222653
4       84844534222
           ...     
174     84368776525
175    817035351368
176     84969663003
177     84943111987
178     84943111987
Name: Phone_clean, Length: 179, dtype: object

In [66]:
df_leads = pd.read_csv(
    "./test/leads_clean_with_note_phone.csv",
    dtype={"Phone_clean": str, "Phone_extracted_from_note": str}
)

In [80]:
df_gmv = pd.read_csv("./test/gmv_clean_sales_infor.csv")

In [73]:
df_leads["Phone_clean"] = (
    df_leads["Phone_clean"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

In [74]:
df_leads["Phone_clean"]

0               <NA>
1        84919249237
2        84965468525
3        84985951781
4        84972856556
            ...     
16331    84909693803
16332    84986806948
16333    84981971660
16334    84389479460
16335    84935030790
Name: Phone_clean, Length: 16336, dtype: string

In [75]:
# import pandas as pd
import re
from difflib import get_close_matches

# =====================================================
# STEP 1. Normalize phone
# =====================================================

def norm_phone(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\D", "", x)  # keep digits only
    return x


df_unmatched["Phone_clean"] = df_unmatched["Phone_clean"].apply(norm_phone)
df_leads["Phone_clean"] = df_leads["Phone_clean"].apply(norm_phone)

# remove empty
unmatched_phones = df_unmatched["Phone_clean"].dropna().unique().tolist()
lead_phones = df_leads["Phone_clean"].dropna().unique().tolist()

# =====================================================
# STEP 2. Fuzzy match using difflib
# =====================================================

matches = []

for p in unmatched_phones:

    if p == "":
        continue

    # lấy 5 match gần nhất
    close = get_close_matches(p, lead_phones, n=5, cutoff=0.80)

    for c in close:
        matches.append({
            "unmatched_phone": p,
            "lead_phone": c
        })

# =====================================================
# STEP 3. Convert to DataFrame
# =====================================================

df_fuzzy = pd.DataFrame(matches)

print("Total fuzzy matches:", len(df_fuzzy))
df_fuzzy.head(30)

Total fuzzy matches: 498


,unmatched_phone,lead_phone
0,84908224672,84983224462
1,84908224672,84982467725
2,84908224672,84977082467
3,84908224672,84968922467
4,84908224672,84902264612
5,84909517679,84909572693
6,84909517679,84909526379
7,84909517679,84908501679
8,84909517679,84904976789
9,84909517679,84904517691


In [79]:
import editdistance

matches = []

for p1 in unmatched_phones:
    if p1 == "":
        continue

    for p2 in lead_phones:

        if abs(len(p1) - len(p2)) <= 2:

            dist = editdistance.eval(p1, p2)

            if dist <= 2:
                matches.append({
                    "unmatched_phone": p1,
                    "lead_phone": p2,
                    "distance": dist
                })

df_fuzzy = pd.DataFrame(matches)
df_fuzzy.sort_values("distance").head(30)

,unmatched_phone,lead_phone,distance
20,84964166015,84964166016,1
12,84966645999,84966645995,1
8,84909610282,84909110282,1
22,84373406103,84334061503,2
23,84944022155,84904022135,2
24,84975567075,84975557095,2
25,84965272428,84965282421,2
26,84984909000,84989809000,2
27,84984909000,84984929006,2
0,84844534222,84844936222,2


In [77]:
# =====================================================
# STEP 1. Clean UID
# =====================================================

def norm_uid(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

df_unmatched["UID_clean"] = df_unmatched["UID_clean"].apply(norm_uid)
df_leads["UID_clean"] = df_leads["UID_clean"].apply(norm_uid)

unmatched_uids = df_unmatched["UID_clean"].dropna().unique().tolist()
lead_uids = df_leads["UID_clean"].dropna().unique().tolist()

# =====================================================
# STEP 2. Fuzzy match (edit distance)
# =====================================================

matches = []

for u1 in unmatched_uids:

    if u1 == "":
        continue

    for u2 in lead_uids:

        # lọc nhanh để giảm cost
        if abs(len(u1) - len(u2)) <= 2:

            dist = editdistance.eval(u1, u2)

            if dist <= 1:
                matches.append({
                    "unmatched_uid": u1,
                    "lead_uid": u2,
                    "distance": dist
                })

# =====================================================
# STEP 3. Result
# =====================================================

df_uid_fuzzy = pd.DataFrame(matches)
df_uid_fuzzy = df_uid_fuzzy.sort_values("distance")

print("Total fuzzy UID matches:", len(df_uid_fuzzy))
df_uid_fuzzy.head(30)

Total fuzzy UID matches: 17


,unmatched_uid,lead_uid,distance
0,3311919278,3311419278,1
14,3311992414,3311992419,1
13,3312225025,3311225025,1
12,3312172635,3312172633,1
11,3312150282,3312156282,1
10,3311684368,3311684968,1
9,3310023176,3310023177,1
15,3312285639,3312283639,1
8,3311857943,3311857443,1
6,3311360177,3311360173,1


In [99]:
def data_quality_report(df):
    total = len(df)

    # Cùng UID nhưng nhiều Phone
    uid_phone = (
        df.dropna(subset=["UID_clean", "Phone_clean"])
          .groupby("UID_clean")["Phone_clean"]
          .nunique()
    )
    same_uid_diff_phone = df["UID_clean"].isin(uid_phone[uid_phone > 1].index).sum()

    # Cùng Phone nhưng nhiều UID
    phone_uid = (
        df.dropna(subset=["UID_clean", "Phone_clean"])
          .groupby("Phone_clean")["UID_clean"]
          .nunique()
    )
    same_phone_diff_uid = df["Phone_clean"].isin(phone_uid[phone_uid > 1].index).sum()

    # Thiếu UID hoặc Phone
    missing_uid = df["UID_clean"].isna().sum()
    missing_phone = df["Phone_clean"].isna().sum()

    print(f"Total rows: {total}")
    print(f"Same UID, different Phone : {same_uid_diff_phone} ({same_uid_diff_phone/total:.2%})")
    print(f"Same Phone, different UID : {same_phone_diff_uid} ({same_phone_diff_uid/total:.2%})")
    print(f"Missing UID               : {missing_uid} ({missing_uid/total:.2%})")
    print(f"Missing Phone             : {missing_phone} ({missing_phone/total:.2%})")

In [100]:
print("=== LEADS ===")
data_quality_report(df_leads)

print("\n=== GMV ===")
data_quality_report(df_gmv)

=== LEADS ===
Total rows: 16336
Same UID, different Phone : 827 (5.06%)
Same Phone, different UID : 551 (3.37%)
Missing UID               : 0 (0.00%)
Missing Phone             : 0 (0.00%)

=== GMV ===
Total rows: 463
Same UID, different Phone : 6 (1.30%)
Same Phone, different UID : 0 (0.00%)
Missing UID               : 0 (0.00%)
Missing Phone             : 0 (0.00%)


In [101]:
print(df_leads["UID_clean"].dtype)
print(df_gmv["UID_clean"].dtype)

object
object


In [107]:
df_leads["UID_clean"] = df_leads["UID_clean"].astype(str)
df_gmv["UID_clean"] = df_gmv["UID_clean"].astype(str)

In [108]:
uid_check = (
    df_leads[["UID_clean", "Phone_clean"]]
    .merge(
        df_gmv[["UID_clean", "Phone_clean"]],
        on="UID_clean",
        suffixes=("_lead", "_gmv")
    )
)

uid_diff_phone = uid_check[
    uid_check["Phone_clean_lead"] != uid_check["Phone_clean_gmv"]
]

print(len(uid_diff_phone))
uid_diff_phone.head()

0


,UID_clean,Phone_clean_lead,Phone_clean_gmv


In [105]:
df_leads["Phone_clean"] = df_leads["Phone_clean"].astype(str)
df_gmv["Phone_clean"] = df_gmv["Phone_clean"].astype(str)

In [106]:
phone_check = (
    df_leads[["UID_clean", "Phone_clean"]]
    .merge(
        df_gmv[["UID_clean", "Phone_clean"]],
        on="Phone_clean",
        suffixes=("_lead", "_gmv")
    )
)

phone_diff_uid = phone_check[
    phone_check["UID_clean_lead"] != phone_check["UID_clean_gmv"]
]

print(len(phone_diff_uid))
phone_diff_uid.head()

26


,UID_clean_lead,Phone_clean,UID_clean_gmv
14,,817090779689,3300978231
43,3311193217,817090293668,3311228095
47,,817084801084,3311228463
53,,420773055288,3286214118
107,3311627622,84357978861,3311634513


In [109]:
df_leads.columns

Index(['Đầu Ngày', 'Họ và tên', 'Người trực', 'Số điện thoại', 'Tuổi',
       'Quốc Gia', 'Note', 'TVTS', 'Ad_id', 'Nguồn', 'Quốc gia \nQuảng cáo',
       'Mẫu Quảng cáo', 'Mã CRM', 'Sale sau phân', 'Link FB', 'UID', 'TT',
       'Trạng thái', 'Ngày lên học thử', 'Current Binded Sale',
       'Sales in latest system-weighted allocation',
       'Latest system-weighted Allocation Time', 'Allocate reason',
       'Tên sale chatpage', 'KOL', 'Check Ad_ID', 'UID_clean', 'Phone_clean',
       'Phone_extracted_from_note'],
      dtype='object')

In [ ]:
df_leads = pd.read_csv("./test/")

In [112]:
df_leads["Mã CRM"].astype("string").str.len().value_counts().sort_index()

Mã CRM
3    6544
8    9779
Name: count, dtype: Int64

In [165]:
df_test_ov_11_cau = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Test OV-11câu 210426.csv", encoding="utf-8")
df_if_ov_lan_2 = pd.read_csv("./data_test/PalFish - Chatpage 2026 - IF OV  (LẦN2).csv", encoding='utf-8')
df_data_gg = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Data GG.csv", encoding='utf-8')
df_data_gg_gop_ov = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Data GG Gộp OV.csv", encoding='utf-8')
df_ls_if_nt = pd.read_csv("./data_test/PalFish - Chatpage 2026 - LS-IF (VN-Nền tảng).csv", encoding='utf-8')
df_ls_if_ntov = pd.read_csv("./data_test/PalFish - Chatpage 2026 -  LS-IF (VN-Nền tảng OV).csv", encoding='utf-8')
df_tt_check = pd.read_csv("./data_test/PalFish - Chatpage 2026 - TT check.csv", encoding='utf-8')
df_active_lc = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Active LC.csv", encoding='utf-8')
df_tt_110 = pd.read_csv("./data_test/PalFish - Chatpage 2026 - Trang tính110.csv", encoding='utf-8')

In [ ]:
df_test_ov_11_cau["Số đang dùng"]
df_if_ov_lan_2["phone_number"]
df_data_gg["Số điện thoại.1"]
df_data_gg_gop_ov["Số điện thoại.1"]
df_ls_if_nt["Unnamed: 18"]
df_ls_if_ntov["Unnamed: 18"]
df_tt_check["Số điện thoại"]
df_active_lc["Số điện thoại"]
df_tt_110["Sốđiệnthoại"]



0       84-357007993
1       84-974563233
2       84-934049510
3       84-869273022
4       84-913833004
            ...     
5548             NaN
5549             NaN
5550             NaN
5551             NaN
5552             NaN
Name: Sốđiệnthoại, Length: 5553, dtype: object

In [155]:
def show_length_samples(series, n_samples=5):
    samples = (
        series.astype("string")
        .to_frame("value")
        .assign(length=lambda x: x["value"].str.len())
        .groupby("length")["value"]
        .apply(lambda x: x.drop_duplicates().head(n_samples).tolist())
    )

    print(samples)
    return samples

In [157]:
import numpy as np

def normalize_phone_number_v2(series):
    # Convert series to string first
    s = series.astype(str)
    
    # Remove 'nan' strings
    s = s.replace('nan', '')
    
    # Remove trailing '.0' from float conversions
    s = s.str.replace(r'\.0$', '', regex=True)
    
    # Helper function to fix Excel scientific notation (e.g., "5,05E+14")
    def fix_scientific_notation(val):
        val = val.strip()
        if 'E+' in val or 'e+' in val:
            try:
                # Replace comma with dot to allow python float casting (e.g., "5,05" -> "5.05")
                val_float = val.replace(',', '.')
                # Convert to float then to int to expand the scientific notation, then to string
                return str(int(float(val_float)))
            except ValueError:
                return val
        return val
        
    # Apply the scientific notation fix
    s = s.apply(fix_scientific_notation)
    
    # Remove all non-digit characters 
    # This automatically handles 'p:', '+', '-', '\r\n', spaces, etc.
    s = s.str.replace(r'\D', '', regex=True)
    
    # Convert empty strings back to NaN for better missing value handling
    s = s.replace('', np.nan)
    
    return s

In [175]:
df_test_ov_11_cau["Số đang dùng"] = normalize_phone_number_v2(df_test_ov_11_cau["Số đang dùng"])
df_if_ov_lan_2["phone_number"] = normalize_phone_number_v2(df_if_ov_lan_2["phone_number"])
df_data_gg["Số điện thoại.1"] = normalize_phone_number_v2(df_data_gg["Số điện thoại.1"])
df_data_gg_gop_ov["Số điện thoại.1"] = normalize_phone_number_v2(df_data_gg_gop_ov["Số điện thoại.1"])
df_ls_if_nt["Unnamed: 18"] = normalize_phone_number_v2(df_ls_if_nt["Unnamed: 18"])
df_ls_if_ntov["Unnamed: 18"] = normalize_phone_number_v2(df_ls_if_ntov["Unnamed: 18"])
df_tt_check["Số điện thoại"] = normalize_phone_number_v2(df_tt_check["Số điện thoại"])
df_active_lc["Số điện thoại"] = normalize_phone_number_v2(df_active_lc["Số điện thoại"])
df_tt_110["Sốđiệnthoại"] = normalize_phone_number_v2(df_tt_110["Sốđiệnthoại"])

In [134]:
col = "Phone_clean"

samples = (
    df_unmatched.assign(length=df_unmatched[col].astype("string").str.len())
    .groupby("length")[col]
    .apply(lambda x: x.drop_duplicates().head(5).tolist())
)

print(samples)

length
11    [84908224672, 84909517679, 84389970625, 849382...
12    [818051986529, 491631303303, 821057761801, 840...
13    [4917644461422, 4915205315239, 4915751184694, ...
Name: Phone_clean, dtype: object


In [176]:
show_length_samples(df_test_ov_11_cau["Số đang dùng"])
show_length_samples(df_if_ov_lan_2["phone_number"])
show_length_samples(df_data_gg["Số điện thoại.1"])
show_length_samples(df_data_gg_gop_ov["Số điện thoại.1"])
show_length_samples(df_ls_if_nt["Unnamed: 18"])
show_length_samples(df_ls_if_ntov["Unnamed: 18"])
show_length_samples(df_tt_check["Số điện thoại"])
show_length_samples(df_active_lc["Số điện thoại"])
show_length_samples(df_tt_110["Sốđiệnthoại"])

length
9                                           [844842845]
10                             [8121676936, 8439618954]
11    [84981655339, 84762767717, 84934426311, 849047...
12    [420774151911, 817084769177, 821079121296, 818...
13    [8494646434343, 8184971287168, 8184989576302, ...
14    [84420601107860, 81420731219486, 8481906623937...
15    [844915730758827, 420818089751945, 81861863835...
16                                   [8449151458355394]
19                                [8109153184564281643]
Name: value, dtype: object
length
5                                               [77555]
11    [84968575161, 84981131382, 84965249362, 210578...
12    [818062639718, 821031363389, 821080072455, 818...
13    [8407894568899, 4915730758827, 4917658123154, ...
Name: value, dtype: object
length
3                                            [819, 812]
4                                          [8181, 8163]
5                                        [49234, 81212]
6                            

length
6                                              [102976]
7         [3794090, 9814531, 9774956, 7827651, 9661515]
8     [93093093, 34933620, 36234551, 50512312, 34892...
9                     [892166821, 456877777, 769759863]
10    [8352211888, 1234567890, 9972172959, 447926640...
11    [84357007993, 84974563233, 84934049510, 848692...
12                         [658894304795, 842255888888]
14    [56500000000000, 53100000000000, 9740000000000...
15    [505000000000000, 995000000000000, 68300000000...
16                                   [1390000000000000]
Name: value, dtype: object

In [177]:
df_unmatched["Phone_clean"]

0       84908224672
1       84909517679
2       84389970625
3       84938222653
4       84844534222
           ...     
174     84368776525
175    817035351368
176     84969663003
177     84943111987
178     84943111987
Name: Phone_clean, Length: 179, dtype: object

In [168]:
def check_intersection(base_series, target_series, name=None):
    base = (
        base_series.astype("string")
        .str.strip()
        .dropna()
    )

    target = (
        target_series.astype("string")
        .str.strip()
        .dropna()
    )

    inter = set(base) & set(target)

    if name is None:
        name = target_series.name

    print(f"{name}: {len(inter)} giá trị trùng")

    if inter:
        print("Ví dụ:", list(sorted(inter))[:10])

    return inter

In [178]:
base = df_unmatched["Phone_clean"]

check_intersection(base, df_test_ov_11_cau["Số đang dùng"])
check_intersection(base, df_if_ov_lan_2["phone_number"])
check_intersection(base, df_data_gg["Số điện thoại.1"])
check_intersection(base, df_data_gg_gop_ov["Số điện thoại.1"])
check_intersection(base, df_ls_if_nt["Unnamed: 18"])
check_intersection(base, df_ls_if_ntov["Unnamed: 18"])
check_intersection(base, df_tt_check["Số điện thoại"])
check_intersection(base, df_active_lc["Số điện thoại"])
check_intersection(base, df_tt_110["Sốđiệnthoại"])

Số đang dùng: 0 giá trị trùng
phone_number: 0 giá trị trùng
Số điện thoại.1: 0 giá trị trùng
Số điện thoại.1: 0 giá trị trùng
Unnamed: 18: 0 giá trị trùng
Unnamed: 18: 0 giá trị trùng
Số điện thoại: 2 giá trị trùng
Ví dụ: ['84909974812', '84986713269']
Số điện thoại: 6 giá trị trùng
Ví dụ: ['818062508538', '84392218155', '84933889568', '84944022155', '84966277873', '84974282165']
Sốđiệnthoại: 1 giá trị trùng
Ví dụ: ['84909974812']


{'84909974812'}

In [182]:
import pandas as pd
import editdistance

def find_phone_close(base_series, target_series, max_dist=2):
    # Chuẩn hóa
    base = (
        base_series.astype("string")
        .dropna()
        .str.strip()
        .drop_duplicates()
    )

    target = (
        target_series.astype("string")
        .dropna()
        .str.strip()
        .drop_duplicates()
    )

    results = []

    target_list = target.tolist()

    for p1 in base:
        if p1 == "":
            continue

        for p2 in target_list:
            if p2 == "":
                continue

            # Bỏ qua nếu chênh lệch độ dài quá lớn
            if abs(len(p1) - len(p2)) > max_dist:
                continue

            dist = editdistance.eval(p1, p2)

            if dist <= max_dist:
                results.append({
                    "base_phone": p1,
                    "target_phone": p2,
                    "edit_distance": dist
                })

    if len(results) == 0:
        return pd.DataFrame(
            columns=["base_phone", "target_phone", "edit_distance"]
        )

    return (
        pd.DataFrame(results)
        .sort_values(["edit_distance", "base_phone"])
        .reset_index(drop=True)
    )


base = df_unmatched["Phone_clean"]

targets = {
    "df_test_ov_11_cau": df_test_ov_11_cau["Số đang dùng"],
    "df_if_ov_lan_2": df_if_ov_lan_2["phone_number"],
    "df_data_gg": df_data_gg["Số điện thoại.1"],
    "df_data_gg_gop_ov": df_data_gg_gop_ov["Số điện thoại.1"],
    "df_ls_if_nt": df_ls_if_nt["Unnamed: 18"],
    "df_ls_if_ntov": df_ls_if_ntov["Unnamed: 18"],
    "df_tt_check": df_tt_check["Số điện thoại"],
    "df_active_lc": df_active_lc["Số điện thoại"],
    "df_tt_110": df_tt_110["Sốđiệnthoại"],
}

for name, series in targets.items():
    print(f"\n{'='*60}")
    print(name)

    result = find_phone_close(base, series, max_dist=2)

    if result.empty:
        print("Không tìm thấy.")
    else:
        print(f"Tìm thấy {len(result)} cặp gần đúng.")
        display(result.head(30))   # Nếu dùng Jupyter
        # print(result.head(20))   # Nếu không dùng Jupyter


df_test_ov_11_cau
Không tìm thấy.

df_if_ov_lan_2
Không tìm thấy.

df_data_gg
Tìm thấy 1 cặp gần đúng.


,base_phone,target_phone,edit_distance
0,84968979682,84969979642,2



df_data_gg_gop_ov
Không tìm thấy.

df_ls_if_nt
Tìm thấy 4 cặp gần đúng.


,base_phone,target_phone,edit_distance
0,84373406103,84373408102,2
1,84908988077,84908588078,2
2,84986203686,84862037686,2
3,84988844456,84908844426,2



df_ls_if_ntov
Tìm thấy 1 cặp gần đúng.


...

,base_phone,target_phone,edit_distance
0,84909974812,84909974812,0
1,84986713269,84986713269,0
2,84908489960,84908989961,2
3,84911331626,84911316826,2
4,84965272428,84975672428,2
5,84971721290,84931721390,2



df_active_lc
Tìm thấy 7 cặp gần đúng.


,base_phone,target_phone,edit_distance
0,818062508538,818062508538,0
1,84392218155,84392218155,0
2,84933889568,84933889568,0
3,84944022155,84944022155,0
4,84966277873,84966277873,0
5,84974282165,84974282165,0
6,84965691407,84965696417,2



df_tt_110
Tìm thấy 5 cặp gần đúng.


,base_phone,target_phone,edit_distance
0,84909974812,84909974812,0
1,84812168889,84812688891,2
2,84908489960,84908989961,2
3,84965272428,84975672428,2
4,84971721290,84931721390,2


### - Phase 1:
#### Sử dụng cách trường cơ bản UID+Phone, Phone, UID của toàn bộ các sheet trong link leads: xử lý được 291/463 (62.8%) dòng
### - Phase 2: 
#### Lấy số điện thoại từ note xử lý thêm được 3/463 (0.6%) dòng
### - Còn lại 169 (36.5 %) chưa xử lý được, cả uid và sdt đều không tồn tại trong lead, sau khi sử dụng edit distance thì phát hiện 17 uid của gmv và lead chỉ lệch 1 đơn vị ví dụ: 3311919278 || 3311419278. 
### 3 phone của gmv và lead chỉ lệch 1 đơn vị ví dụ: 84964166015 || 84964166016 liệu có thể cân nhắc sử dụng hay không?
### Ngoài ra xem xét còn có thể sử dụng các trường dữ liệu nào khác ngoài các trường cơ bản đã nêu trên?
